# Error suppression on a real quantum computer

Companion notebook to **Lattice Atlas** — the last rung of the verification ladder:

1. **Browser Lab** — the toy model: watch curves separate at ~15% (code-capacity noise)
2. **Stim + PyMatching** (`first-threshold-curve.ipynb`) — the research simulator: crossing drops to ~1% (circuit noise)
3. **This notebook** — the same claim, *measured on actual superconducting qubits*: bigger codes protect better.

We run a **bit-flip repetition code memory** — the QEC hello-world — on IBM's free Open Plan hardware (127+-qubit Heron processors, no payment or application needed: create an account at https://quantum.cloud.ibm.com). Honest scoping up front: this is a repetition code (protects one error type, no repeated syndrome extraction), not a surface code — Google's Willow below-threshold experiment is the full version of what you measure here in miniature.

In [ ]:
%pip install -q qiskit qiskit-aer qiskit-ibm-runtime matplotlib numpy

## 1 · The experiment

Hold logical |1⟩ = |1…1⟩ on a chain of n qubits through a stretch of idle time (built from X-gate pairs — the barriers stop the transpiler from cancelling them), then measure and **decode by majority vote**. Amplitude damping (T1 decay) flips qubits toward |0⟩; the vote outvotes minority flips. A logical error = the majority flipped.

Prediction from the theory you learned on the site: logical error should **fall as n grows** — same scaling law as the Lab's chart, on hardware.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

def repetition_memory(n_qubits: int, idle_pairs: int) -> QuantumCircuit:
    qc = QuantumCircuit(n_qubits, n_qubits)
    qc.x(range(n_qubits))
    qc.barrier()
    for _ in range(idle_pairs):
        qc.x(range(n_qubits))
        qc.barrier()
        qc.x(range(n_qubits))
        qc.barrier()
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

def logical_error_rate(counts: dict, n_qubits: int):
    shots = sum(counts.values())
    fails = sum(c for bits, c in counts.items() if bits.count('1') <= n_qubits // 2)
    return fails / shots, shots

DISTANCES = [3, 5, 7]
IDLE_PAIRS = 24
SHOTS = 4000  # gentle on your free minutes; raise for tighter error bars

## 2 · Dry run on the local simulator first

Same circuits, thermal-relaxation noise model. Expect roughly: d=3 ≈ 1.8%, d=5 ≈ 0.5%, d=7 ≈ 0.1% (Λ ≈ 4–5 per step). If this cell shows suppression, your pipeline is correct and hardware time won't be wasted.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error

noise = NoiseModel()
noise.add_all_qubit_quantum_error(thermal_relaxation_error(300.0, 200.0, 1.0), ['x'])
sim = AerSimulator(noise_model=noise)

sim_rates = {}
for n in DISTANCES:
    counts = sim.run(repetition_memory(n, IDLE_PAIRS), shots=20_000).result().get_counts()
    rate, shots = logical_error_rate(counts, n)
    sim_rates[n] = rate
    print(f'd={n}: simulated logical error {rate:.4f}')
assert sim_rates[5] < sim_rates[3] and sim_rates[7] < sim_rates[5]
print('suppression confirmed on simulator — safe to spend hardware minutes')

## 3 · The real thing

Get your (free) API token from https://quantum.cloud.ibm.com → paste below once; `save_account` stores it locally. The transpiler maps each chain onto the device's best-connected, least-noisy physical qubits.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler import generate_preset_pass_manager

# First time only — then you can delete this line:
# QiskitRuntimeService.save_account(token='YOUR_TOKEN', set_as_default=True)

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)
print(f'running on {backend.name} ({backend.num_qubits} qubits)')

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuits = [pm.run(repetition_memory(n, IDLE_PAIRS)) for n in DISTANCES]

sampler = SamplerV2(mode=backend)
job = sampler.run(isa_circuits, shots=SHOTS)
print(f'job id: {job.job_id()} — waiting for the quantum computer…')
result = job.result()

In [ ]:
import matplotlib.pyplot as plt

hw_rates, hw_ci = {}, {}
for n, res in zip(DISTANCES, result):
    counts = res.data.c.get_counts()
    rate, shots = logical_error_rate(counts, n)
    hw_rates[n] = rate
    hw_ci[n] = 1.96 * np.sqrt(rate * (1 - rate) / shots) if rate > 0 else 3 / shots
    print(f'd={n}: MEASURED logical error {rate:.4f} ± {hw_ci[n]:.4f}')

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.errorbar(DISTANCES, [sim_rates[n] for n in DISTANCES], marker='s', ls='--',
            color='#0891B2', label='simulator (relaxation model)')
ax.errorbar(DISTANCES, [hw_rates[n] for n in DISTANCES],
            yerr=[hw_ci[n] for n in DISTANCES], marker='o',
            color='#D97706', capsize=4, label=f'REAL HARDWARE ({backend.name})')
ax.set_yscale('log')
ax.set_xticks(DISTANCES)
ax.set_xlabel('code distance (chain length)')
ax.set_ylabel('logical error rate')
ax.set_title('Bigger codes win — measured on actual qubits')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

if hw_rates[5] > 0 and hw_rates[7] > 0:
    print(f'Λ(3→5) = {hw_rates[3]/hw_rates[5]:.1f} · Λ(5→7) = {hw_rates[5]/hw_rates[7]:.1f}')
print('Compare with Google Willow (surface code, full syndrome extraction): Λ ≈ 2.1 — arXiv:2408.13687')

## 4 · What you just verified — and what you didn't

**Verified end-to-end**: redundancy + decoding suppresses logical errors on real quantum hardware, and suppression grows with distance — the claim behind every chart on the site, now backed by your own measurement.

**Honest gaps between this and full QEC** (each is a topic on the site):
- The repetition code protects only against bit flips — a surface code protects both X and Z (*quantum codes basics*).
- We measured once at the end; real codes measure syndromes every microsecond, forever, and must decode measurement errors too (*syndrome extraction*, `first-threshold-curve.ipynb` covers this in simulation).
- Majority vote is the decoder here; surface codes need matching (*decoding/MWPM* — play Decoder Duel to feel why).

### Keep going
- Rerun at different `IDLE_PAIRS` — find where suppression breaks down as noise accumulates.
- Try `n = 9, 11` — does Λ hold?
- Advanced: add mid-circuit syndrome measurements with ancillas between data qubits (Heron supports dynamic circuits) — that's one step from a real surface-code experiment.
- Willow-scale versions of this experiment: Google's Early Access Program takes proposals (rounds announced at quantum.ai.google) — this notebook is the seed of one.